[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/fingerprint-similarity-guide/blob/main/notebooks/12_decision_guide.ipynb)

# 12 · The Decision Guide

*Fingerprint & Similarity Guide · Hamza Hentabli, Ph.D. · ~35 min*

You have run four real benchmarks. This final notebook turns them into a **cheat sheet**: for each task, which fingerprint + similarity metric to reach for, backed by the numbers you computed. Pin this one.

**Learning objectives.** By the end of this notebook you will be able to:

- read a single decision table mapping task → recommended fingerprint + metric
- see the real winning scores behind each recommendation
- understand the one trade-off that drives all of it (specificity vs tolerance)
- avoid the default-everything-to-ECFP4+Tanimoto reflex when it is wrong

In [1]:
# Setup: on Google Colab this installs dependencies and clones the repo so the
# cached data and the shared `fsg` helper module are available. Running locally,
# install requirements.txt once and this cell just locates the files.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
OFFLINE = os.environ.get("FSG_OFFLINE", "0") == "1"   # CI sets this; no notebook needs the network
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rdkit", "scikit-learn", "scipy", "pandas", "matplotlib"], check=True)
    if not os.path.isdir("fingerprint-similarity-guide"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/fingerprint-similarity-guide.git"], check=True)
    ROOT = "fingerprint-similarity-guide"
else:
    ROOT = ".." if os.path.isdir("../data") else "."
DATA_DIR = os.path.join(ROOT, "data")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import fsg   # shared fingerprint factory + metrics + evaluation

print("Colab:", IN_COLAB, "| offline:", OFFLINE, "| data:", DATA_DIR)
print("Fingerprints available:", ", ".join(fsg.FP_NAMES))

Colab: False | offline: True | data: ../data
Fingerprints available: ECFP4-1024, ECFP4-2048, ECFP6-2048, ECFP4-counts, FCFP4-2048, RDKit, AtomPair, Torsion, MACCS, Avalon


In [2]:
import json, os
import pandas as pd
DOCS_DIR = os.path.join(ROOT, "docs")
RESULTS = json.load(open(os.path.join(DOCS_DIR, "benchmark_results.json")))
print("Loaded committed benchmark results from docs/benchmark_results.json")

Loaded committed benchmark results from docs/benchmark_results.json


## 1. The winners, pulled straight from the benchmark results

In [3]:
b1 = RESULTS["benchmark1_virtual_screening"]["scores"]
b2r = RESULTS["benchmark2_scaffold_hopping"]["retention"]
b4 = RESULTS["benchmark4_knn_qsar_esol"]["scores"]
b3 = RESULTS["benchmark3_activity_cliffs"]["scores"]

vs_best  = max(b1.items(), key=lambda kv: kv[1]["BEDROC"])
qsar_best= max(b4.items(), key=lambda kv: kv[1]["R2"])
hop_best = max(b2r.items(), key=lambda kv: (kv[1]["retention_ratio"] or 0))
# most-specific (fewest spurious near-neighbours) for cliff-aware work
cliff_fewest = min(b3.items(), key=lambda kv: kv[1]["n_near_neighbor_pairs"])
print("Virtual screening  :", vs_best[0], "-> BEDROC", vs_best[1]["BEDROC"], "EF1%", vs_best[1]["EF1%"])
print("Scaffold hopping   :", hop_best[0], "-> cross/same retention", hop_best[1]["retention_ratio"])
print("kNN QSAR (ESOL)    :", qsar_best[0], "-> R2", qsar_best[1]["R2"])
print("Fewest false nbrs  :", cliff_fewest[0], "-> near-nbr pairs", cliff_fewest[1]["n_near_neighbor_pairs"])

Virtual screening  : ECFP4-2048|russel -> BEDROC 0.862 EF1% 6.302
Scaffold hopping   : MACCS -> cross/same retention 0.679
kNN QSAR (ESOL)    : ECFP4-counts|tanimoto -> R2 0.772
Fewest false nbrs  : ECFP6-2048 -> near-nbr pairs 1028


## 2. The cheat sheet

In [4]:
import pandas as pd
guide = pd.DataFrame([
    ["Virtual screening (find analogues of a hit)",
     "ECFP4 (2048) or Torsion", "Tanimoto / Dice",
     f"BEDROC {vs_best[1]['BEDROC']}, EF1% {vs_best[1]['EF1%']} on DUD-E EGFR"],
    ["Scaffold hopping (new core, same activity)",
     "FCFP4 / AtomPair / RDKit (feature or topological)", "Tanimoto / Dice",
     f"best cross-scaffold retention ({hop_best[0]} {hop_best[1]['retention_ratio']}); avoid ECFP6"],
    ["Substructure / scaffold containment",
     "ECFP4", "Tversky (alpha=1, beta=0)",
     "ranks true SMARTS matches far above Tanimoto (Notebook 09)"],
    ["Nearest-neighbour QSAR (regression)",
     "ECFP4-counts or AtomPair", "Tanimoto / Dice",
     f"R2 {qsar_best[1]['R2']} on ESOL; counts > binary; NEVER Russel-Rao"],
    ["Clustering / diversity picking",
     "ECFP4 / FCFP4 (2048, high resolution)", "Tanimoto",
     "coarse fps (MACCS) under-count diversity (Notebook 10)"],
    ["Activity-cliff-aware analysis",
     "ECFP6 (most specific)", "Tanimoto",
     f"fewest spurious near-neighbours ({cliff_fewest[0]}); MACCS invents the most"],
    ["Million-compound search (speed)",
     "ECFP4 (1024-2048), binary", "Tanimoto (popcount)",
     "use bulk / FPSim2 (Notebook 11)"],
], columns=["Task", "Fingerprint", "Metric", "Why (real result)"])
pd.set_option("display.max_colwidth", None)
guide

,Task,Fingerprint,Metric,Why (real result)
0,Virtual screening (find analogues of a hit),ECFP4 (2048) or Torsion,Tanimoto / Dice,"BEDROC 0.862, EF1% 6.302 on DUD-E EGFR"
1,"Scaffold hopping (new core, same activity)",FCFP4 / AtomPair / RDKit (feature or topological),Tanimoto / Dice,best cross-scaffold retention (MACCS 0.679); avoid ECFP6
2,Substructure / scaffold containment,ECFP4,"Tversky (alpha=1, beta=0)",ranks true SMARTS matches far above Tanimoto (Notebook 09)
3,Nearest-neighbour QSAR (regression),ECFP4-counts or AtomPair,Tanimoto / Dice,R2 0.772 on ESOL; counts > binary; NEVER Russel-Rao
4,Clustering / diversity picking,"ECFP4 / FCFP4 (2048, high resolution)",Tanimoto,coarse fps (MACCS) under-count diversity (Notebook 10)
5,Activity-cliff-aware analysis,ECFP6 (most specific),Tanimoto,fewest spurious near-neighbours (ECFP6-2048); MACCS invents the most
6,Million-compound search (speed),"ECFP4 (1024-2048), binary",Tanimoto (popcount),use bulk / FPSim2 (Notebook 11)


## 3. The one idea behind the whole table

Every recommendation is a point on a single trade-off:

- **Specific** fingerprints (ECFP, especially ECFP6; 2,048 bits; counts) separate close analogues sharply → best for **screening, QSAR, diversity, cliff-awareness**.
- **Tolerant** fingerprints (FCFP features, atom pairs, RDKit paths, MACCS keys) keep seeing two molecules as similar across bigger changes → best for **scaffold hopping**, but they blur screening and invent cliffs.

And on metrics: **Tanimoto/Dice/Cosine** are safe and near-interchangeable for good fingerprints; **Tversky** is a specialist for directional/substructure queries; **Russel-Rao** is a trap for prediction. The reflex 'ECFP4 + Tanimoto' is a *fine default* and genuinely wins screening — but it is the wrong tool for scaffold hopping, substructure search, and count-sensitive QSAR.

## 4. Summary figure

![task winners](../docs/task_winners.png)

*Best fingerprint + metric per task, from the benchmarks in this repository (regenerate with `python scripts/make_figures.py`).*

## You're done

You can now justify a fingerprint and metric choice with numbers instead of habit. Re-run any benchmark on **your** target by swapping the dataset in `data/` and re-running `scripts/run_benchmarks.py`. If this helped, a star on the repo and a citation (`CITATION.cff`) are appreciated.

— *Hamza Hentabli, Ph.D.*

### ✏️ Exercise 1

Print your own top recommendation for scaffold hopping by sorting the retention table, excluding MACCS (too coarse for screening).

<details>
<summary><b>Show solution</b></summary>

```python
cands = {k:v["retention_ratio"] for k,v in b2r.items() if k!="MACCS"}
print(sorted(cands.items(), key=lambda kv:-kv[1])[:3])
```

</details>

### ✏️ Exercise 2

For virtual screening, confirm ECFP4-2048+Tanimoto is within 0.02 BEDROC of the overall best combo (i.e. the default is nearly optimal here).

<details>
<summary><b>Show solution</b></summary>

```python
print("default:", b1["ECFP4-2048|tanimoto"]["BEDROC"], "| best:", vs_best[1]["BEDROC"])
```

</details>